# AgriVision-STFNet — Architecture Outline



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.13.0+cpu

## 0. Config

Small, deliberately "basic" channel widths (32 -> 64 -> 128) — this is a first
architecture, not a tuned final one. `base` is the one knob that scales the whole
model up or down.

In [2]:
IN_CHANNELS = 8   # 4 MODIS bands (e.g. Red, Green, Blue, NIR) + 4 Sentinel-2 bands, concatenated
OUT_CHANNELS = 4  # reconstructed Sentinel-2-like bands: Red, Green, Blue, NIR
BASE = 32         # base channel width; encoder goes base -> base*2 -> base*4
NUM_HEADS = 4     # attention heads in the single transformer block

## 1. CNN Encoder

Extracts local spectral-spatial features and downsamples 8x (three stride-2 convs)
before the expensive blocks run. Returns **two** things: the deep `bottleneck`
feature map that the rest of the pipeline operates on, and a full-resolution `skip`
feature map (just the stem output) kept aside for the Feature Fusion stage later —
the same role a U-Net skip connection plays.

In [3]:
class CNNEncoder(nn.Module):
    def __init__(self, in_channels, base=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True),
        )
        self.down1 = self._down(base, base * 2)      # H   -> H/2
        self.down2 = self._down(base * 2, base * 4)  # H/2 -> H/4
        self.down3 = self._down(base * 4, base * 4)  # H/4 -> H/8

    @staticmethod
    def _down(cin, cout):
        return nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=2, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        skip = self.stem(x)      # (B, base, H, W)      -- kept for Feature Fusion
        x = self.down1(skip)
        x = self.down2(x)
        bottleneck = self.down3(x)  # (B, base*4, H/8, W/8)
        return bottleneck, skip

## 2. Residual Blocks

Standard two-conv residual block with a skip-add. Directly mirrors the residual
dense block used in MARSTFN (paper #2 in the SLR) — simplified to a plain residual
block (no dense inter-layer connections) to keep parameter count down.

In [4]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

## 3. Multiscale Module

MODIS (250 m) is ~25x coarser than Sentinel-2 (10 m), and Punjab field sizes are
2-10 ha (SRS S1.2) -- no single kernel size sees both "the whole coarse pixel" and
"this one field's edge" at once. Four parallel branches (1x1, 3x3, dilated-3x3,
global-pooled) give several effective receptive fields in one block, ASPP-style.
Matches MARSTFN's named "multiscale mechanism" (SLR paper #2).

In [5]:
class MultiScaleModule(nn.Module):
    def __init__(self, channels, branch_channels):
        super().__init__()
        b = branch_channels
        self.branch1 = nn.Sequential(nn.Conv2d(channels, b, 1), nn.BatchNorm2d(b), nn.ReLU(inplace=True))
        self.branch2 = nn.Sequential(nn.Conv2d(channels, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(inplace=True))
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, b, 3, padding=2, dilation=2), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
        )
        self.branch4 = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, b, 1), nn.ReLU(inplace=True))
        self.merge = nn.Sequential(nn.Conv2d(b * 4, channels, 1), nn.BatchNorm2d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        h, w = x.shape[-2:]
        b1, b2, b3 = self.branch1(x), self.branch2(x), self.branch3(x)
        b4 = F.interpolate(self.branch4(x), size=(h, w), mode="nearest")
        return self.merge(torch.cat([b1, b2, b3, b4], dim=1))

## 4. Transformer Bottleneck (single block)

Self-attention is O((H*W)^2) in the number of spatial tokens, so it only runs here
-- at the encoder's 8x-downsampled bottleneck, where a 256x256 input becomes 32x32 =
1,024 tokens instead of 65,536 (~4,096x cheaper than full-resolution attention for
the same "look anywhere in the scene" capability). One block, not a full ViT stack --
grounded in SLR paper #13 (target self-attention STF) and #7 (CRFNet's multi-head
attention hybrid).

In [6]:
class TransformerBottleneck(nn.Module):
    def __init__(self, channels, num_heads=4, mlp_ratio=2.0, max_tokens=4096):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(channels)
        hidden = int(channels * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(channels, hidden), nn.GELU(), nn.Linear(hidden, channels))
        # learned positional embedding, sized for the largest patch we expect to train on
        self.pos_embed = nn.Parameter(torch.zeros(1, max_tokens, channels))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        b, c, h, w = x.shape
        n = h * w
        tokens = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        assert n <= self.pos_embed.shape[1], "increase max_tokens for this patch size"
        tokens = tokens + self.pos_embed[:, :n, :]
        normed = self.norm1(tokens)
        attn_out, _ = self.attn(normed, normed, normed)
        tokens = tokens + attn_out
        tokens = tokens + self.mlp(self.norm2(tokens))
        return tokens.transpose(1, 2).reshape(b, c, h, w)

## 5. CBAM (channel + spatial attention)

Cheap complementary attention (<1% of model parameters, Woo et al. 2018) used
*instead of* a second transformer block -- CBAM handles "which channels / which
pixels matter" re-weighting everywhere in the network at near-zero cost, while the
one expensive self-attention block above is reserved for genuinely non-local
reasoning. Directly grounded in MARSTFN's CSARDB module (SLR paper #2), which is
CBAM fused with a residual block.

In [7]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1), nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(F.adaptive_avg_pool2d(x, 1))
        max_out = self.mlp(F.adaptive_max_pool2d(x, 1))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * attn


class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel_attn = ChannelAttention(channels)
        self.spatial_attn = SpatialAttention()

    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

## 6. Decoder upsampling + Feature Fusion

The decoder mirrors the encoder (three x2 upsampling stages, undoing the 8x
downsample). **Implementation note:** the "Feature Fusion" stage from the pipeline
diagram conceptually sits *before* the decoder, but a skip-connection fusion needs
both feature maps at the *same spatial resolution* -- so in code it necessarily runs
once the deep path has been upsampled back up to the skip's resolution, i.e.
interleaved with decoding rather than strictly before it. It's kept as its own
`FeatureFusion` class so the pipeline's conceptual stages stay one-to-one with the
code.

In [8]:
class DecoderUpsampler(nn.Module):
    """Undoes the encoder's 8x downsample. Outputs `base` channels at full (H, W)
    resolution -- same shape as the encoder's `skip` feature map, ready to fuse."""

    def __init__(self, in_channels):
        super().__init__()
        c = in_channels
        self.up1 = self._up(c, c // 2)       # H/8 -> H/4
        self.up2 = self._up(c // 2, c // 4)  # H/4 -> H/2
        self.up3 = self._up(c // 4, c // 4)  # H/2 -> H

    @staticmethod
    def _up(cin, cout):
        return nn.Sequential(
            nn.ConvTranspose2d(cin, cout, 4, stride=2, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.up3(self.up2(self.up1(x)))


class FeatureFusion(nn.Module):
    """Merges the CBAM-refined 'deep' path with the shallow encoder 'skip' path,
    both now at the same (H, W) resolution -- a 1x1 conv after concatenation, the
    same role a U-Net skip connection plays."""

    def __init__(self, deep_channels, skip_channels, out_channels):
        super().__init__()
        self.merge = nn.Sequential(
            nn.Conv2d(deep_channels + skip_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, deep, skip):
        return self.merge(torch.cat([deep, skip], dim=1))

## 7. Assembling AgriVisionSTFNet

Each pipeline stage is exactly one line in `forward()`, in the same order as the
proposed diagram -- deliberately, so the mapping from "the pipeline we proposed" to
"the code that implements it" stays obvious and each block stays swappable (SDS
S3.1.5 treats the model as a replaceable component).

**NDVI is not a learned head.** The final conv ends in a `Sigmoid`, so the
reconstructed Red/NIR channels are bounded to `[0, 1]` by construction. NDVI is then
computed directly from those two channels with the standard formula. Algebraically,
if `Red, NIR in [0, 1]`, the result is guaranteed to land inside `(-1, 1)` -- which is
exactly the SRS's NDVI range requirement -- with zero extra parameters and zero
clamping logic.

In [9]:
class AgriVisionSTFNet(nn.Module):
    def __init__(self, in_channels=8, base=32, out_channels=4, num_heads=4):
        super().__init__()
        self.encoder = CNNEncoder(in_channels, base)
        self.res_blocks = nn.Sequential(ResBlock(base * 4), ResBlock(base * 4))
        self.multiscale = MultiScaleModule(base * 4, branch_channels=base)
        self.transformer = TransformerBottleneck(base * 4, num_heads=num_heads)
        self.cbam = CBAM(base * 4)
        self.decoder_up = DecoderUpsampler(base * 4)
        self.fusion = FeatureFusion(deep_channels=base, skip_channels=base, out_channels=base)
        self.head = nn.Sequential(nn.Conv2d(base, out_channels, 3, padding=1), nn.Sigmoid())

    def forward(self, modis, sentinel2_ref):
        """
        modis, sentinel2_ref: (B, 4, H, W) each, both already on the same 10 m grid
        (MODIS resampled upstream -- see ml-pipeline/preprocessing/). Bands assumed
        ordered [Red, Green, Blue, NIR].
        """
        x = torch.cat([modis, sentinel2_ref], dim=1)   # early fusion: (B, 8, H, W)

        bottleneck, skip = self.encoder(x)              # 1. CNN Encoder
        feat = self.res_blocks(bottleneck)               # 2. Residual Blocks
        feat = self.multiscale(feat)                      # 3. Multiscale Module
        feat = self.transformer(feat)                     # 4. Transformer
        feat = self.cbam(feat)                            # 5. CBAM
        deep = self.decoder_up(feat)                      # 6a. CNN Decoder (upsample)
        fused = self.fusion(deep, skip)                   # 6b. Feature Fusion (skip merge)
        reflectance = self.head(fused)                    # 7. output head -> (B, 4, H, W) in [0, 1]

        red, nir = reflectance[:, 0:1], reflectance[:, 3:4]
        ndvi = (nir - red) / (nir + red + 1e-6)           # 8. NDVI -- computed, not learned

        return {"reflectance": reflectance, "ndvi": ndvi}

## 8. Smoke test

Not a real training or evaluation run -- just proof the shapes line up end to end,
a parameter-count breakdown per stage (the cost story from the README, as real
numbers instead of estimates), and a check that the NDVI range guarantee actually
holds.

In [10]:
model = AgriVisionSTFNet(in_channels=IN_CHANNELS, base=BASE, out_channels=OUT_CHANNELS, num_heads=NUM_HEADS)

B, H, W = 2, 256, 256

modis = torch.rand(B, 4, H, W)          # dummy -- stand-in for preprocessed MODIS bands
sentinel2_ref = torch.rand(B, 4, H, W)  # dummy -- stand-in for the reference Sentinel-2 tile

with torch.no_grad():
    out = model(modis, sentinel2_ref)

print("reflectance:", tuple(out["reflectance"].shape),
      f"range=[{out['reflectance'].min():.3f}, {out['reflectance'].max():.3f}]")
print("ndvi:       ", tuple(out["ndvi"].shape),
      f"range=[{out['ndvi'].min():.3f}, {out['ndvi'].max():.3f}]")

assert out["ndvi"].min() >= -1.0 and out["ndvi"].max() <= 1.0
print("\nNDVI range check passed: output stays within [-1, 1] with no clamping.")

reflectance: (2, 4, 256, 256) range=[0.126, 0.847]
ndvi:        (2, 1, 256, 256) range=[-0.564, 0.526]

NDVI range check passed: output stays within [-1, 1] with no clamping.

In [11]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}  (~{total_params * 4 / 1e6:.1f} MB as float32)\n")

print(f"{'stage':<15}{'params':>12}")
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:<15}{n:>12,}")

Total parameters: 1,778,359  (~7.1 MB as float32)

stage                params
encoder             242,976
res_blocks          591,360
multiscale           99,008
transformer         656,768
cbam                  4,339
decoder_up          180,608
fusion                2,144
head                  1,156

## 9. Next steps

This notebook stops at "the architecture runs on dummy tensors." What comes before
and after it, per `PROJECT_STATUS.md`'s own build order:

1. `ml-pipeline/data_acquisition/` + `ml-pipeline/preprocessing/` -- real paired
   MODIS/Sentinel-2 NDVI tiles, which is what actually fixes `IN_CHANNELS`, patch
   size, and band order used above (placeholders for now).
2. A dataset/dataloader and a training loop (`ml-pipeline/model/training/`) with a
   simple reconstruction loss (e.g. L1 on `reflectance`, optionally + weighted L1 on
   `ndvi`) -- no GAN, no adversarial loss (see README S4, point 5: the SRS's
   determinism requirement argues against it).
3. Benchmarking against the classical baselines (STARFM/FSDAF) this project's SLR
   also covers, per `ml-pipeline/evaluation/`.
4. Documented, not-yet-built extensions: environmental covariates as extra input
   channels (SLR paper #10); a dual-branch encoder with learned cross-attention if
   early fusion underperforms; a deeper transformer stack if one block proves
   insufficient for long-range context.